# **Assignment 5: Accuracy Measures**

## Author: Mohammad Mehdi Zare, Mohammad Ali Olama

# Abstract

In this assignment, we are going to calculate different metrics for classification and regression tasks in machine learning.

# Libraries

In [99]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [100]:
!pip install arabic_reshaper

In [101]:
!pip install python-bidi

In [102]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings as wr
from matplotlib.category import UnitData

# We need these libraries for Farsi writting
import arabic_reshaper
import bidi.algorithm

wr.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Regression


## **Read dataset**

Here, we are going to perform a regression task on [Real estate price prediction](https://www.kaggle.com/datasets/quantbruce/real-estate-price-prediction) dataset.

first let's read it :

In [103]:
df_org=pd.read_csv("/content/Real estate.csv" , index_col=0)

In [104]:
df = df_org
df.head()

,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
No,,,,,,,
1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
2,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
3,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
4,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
5,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1


In [105]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 414 entries, 1 to 414
Data columns (total 7 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   X1 transaction date                     414 non-null    float64
 1   X2 house age                            414 non-null    float64
 2   X3 distance to the nearest MRT station  414 non-null    float64
 3   X4 number of convenience stores         414 non-null    int64  
 4   X5 latitude                             414 non-null    float64
 5   X6 longitude                            414 non-null    float64
 6   Y house price of unit area              414 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 25.9 KB


Our task is to predict 'Y house price of unit area'.

First, let's split the dataset:

In [106]:
from sklearn.model_selection import train_test_split

X = df.drop('Y house price of unit area', axis=1)
y = df['Y house price of unit area']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [107]:
#training linear regression model and evaluate metrics
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

r2 = r2_score(y_test, y_pred)


print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Mean Absolute Percentage Error (MAPE):", mape)
print("R-squared:", r2)

Mean Absolute Error (MAE): 5.305355690074184
Mean Squared Error (MSE): 53.505619124504506
Mean Absolute Percentage Error (MAPE): 17.182519158883537
R-squared: 0.6810580555095801


We think that the MSE is in a reasonable range, and our model is doing a good job. However, we can improve our accuracy by changing our model to a more complex model like RandomForestRegressor.

# Binary classification

Here, we are going to perform a regression task on [breast cancer](https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset) dataset.

Here our target variable is 'diagnosis'. What we want to do is to predict wether our patient has cancer or not (M - Malignant B - Benign).

first let's read the data :

In [108]:
df_org=pd.read_csv("/content/breast-cancer.csv" , index_col=0)
df = df_org
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


Now, perform train test split :

In [109]:
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Since our task is binary classification, Decision tree or logisitic regression would be good choices. Here we choose logistic regression:

In [110]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#  define model and train it
model = LogisticRegression()
model.fit(X_train, y_train)

# evaluate on test data
y_pred = model.predict(X_test)


# calculate Accuracy measures
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='M')
recall = recall_score(y_test, y_pred, pos_label='M')
f1 = f1_score(y_test, y_pred, pos_label='M')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Accuracy: 0.956140350877193
Precision: 0.975
Recall: 0.9069767441860465
F1-score: 0.9397590361445783


Our results are good! F1-score is a more reliable metric (than precision, recall and accuracy). Since our F1-Score is high (near 1), we belive our simple model is performing good, however by doing some fine-tunning we can acheive better results.

# Multi-class classification

Here, we are going to perform a regression task on [Students Performance Dataset](https://www.kaggle.com/datasets/rabieelkharoua/students-performance-dataset) dataset.

Here our task is to predict grade of students (5 classes: from 0 to 4) based on other variables.

first let's read the dataset :

In [111]:
df_org=pd.read_csv("/content/Student_performance_data _.csv" , index_col=0)
df = df_org
df.head()

,Age,Gender,Ethnicity,ParentalEducation,StudyTimeWeekly,Absences,Tutoring,ParentalSupport,Extracurricular,Sports,Music,Volunteering,GPA,GradeClass
StudentID,,,,,,,,,,,,,,
1001,17,1,0,2,19.833723,7,1,2,0,0,1,0,2.929196,2.0
1002,18,0,0,1,15.408756,0,0,1,0,0,0,0,3.042915,1.0
1003,15,0,2,3,4.210570,26,0,2,0,0,0,0,0.112602,4.0
1004,17,1,0,3,10.028829,14,0,3,1,0,0,0,2.054218,3.0
1005,17,1,0,2,4.672495,17,1,3,0,0,0,0,1.288061,4.0


Our target variable has 5 different classes, which are as follows:

In [112]:
sorted(df['GradeClass'].unique())

[0.0, 1.0, 2.0, 3.0, 4.0]

again, train test split:

In [113]:
X = df.drop('GradeClass', axis=1)
y = df['GradeClass']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

in this part we use decision tree for our multi-class classification task :

In [114]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# define model, train, and predict on test data

model = DecisionTreeClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

to evaluate accuracy measures, the '*classification_report*' function will report almost everything we want!

In [115]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.60      0.55      0.57        22
         1.0       0.66      0.84      0.74        49
         2.0       0.89      0.80      0.84        85
         3.0       0.80      0.80      0.80        86
         4.0       0.91      0.90      0.90       237

    accuracy                           0.84       479
   macro avg       0.77      0.78      0.77       479
weighted avg       0.85      0.84      0.84       479



we can also use other predefined functions to calculate precision, recall and F1.

In [116]:
precision_each_class = precision_score(y_test, y_pred, average=None)
recall_each_class = recall_score(y_test, y_pred, average=None)

f1_macro = f1_score(y_test, y_pred, average='macro')
f1_micro = f1_score(y_test, y_pred, average='micro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print("Precision for each class:", precision_each_class)
print("Recall for each class:", recall_each_class)
print("Macro-averaged F1-score:", f1_macro)
print("Micro-averaged F1-score:", f1_micro)
print("Weighted-averaged F1-score:", f1_weighted)

Precision for each class: [0.6        0.66129032 0.89473684 0.80232558 0.90638298]
Recall for each class: [0.54545455 0.83673469 0.8        0.80232558 0.89873418]
Macro-averaged F1-score: 0.771951152267685
Micro-averaged F1-score: 0.8413361169102297
Weighted-averaged F1-score: 0.8423244496420313


#Final Question
***What accuracy metric do you use to best capture the accuracy of classification algorithm which predicts the above classes based on some data from each player and why?***

In a multi-label classification problem like this one, where each player may belong to multiple independent classes, we might have ***fully-correct***, ***partially-correct***, or ***fully-incorrect*** predictions. There are a lot of metrics to be used, such as accuracy, hamming loss, precision, recall and F1-score.

 F1 score is A good accuracy metric, since it combines precision and recall.
The F1 score balances precision and recall and is particularly useful in cases with ***imbalanced classes***, which is likely here (e.g., fewer players may have heart problems than have played on a national team). You can use either a macro-averaged F1 (treating each label equally regardless of class size) or a weighted F1 score (which accounts for the class distribution). This metric helps capture the effectiveness of the model’s predictions on each class individually and provides an overall picture of the model’s performance.
